# Training Curves

Pull W&B runs and visualize loss curves across all training stages.
Also works with local NeMo log files if W&B is not enabled.

**Stages:** pretrain → sft-chat → sft-code → dpo

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

RESULTS_DIR = Path("/results")
USE_WANDB   = False  # set True if you logged to W&B
WANDB_PROJECT = "slm"

## Option A — W&B (if enabled)

In [ ]:
if USE_WANDB:
    import wandb
    api = wandb.Api()
    runs = api.runs(WANDB_PROJECT)

    print(f"Found {len(runs)} runs in project '{WANDB_PROJECT}':")
    for run in runs:
        print(f"  {run.name:<30} state={run.state}  steps={run.summary.get('_step', '?')}")

In [ ]:
if USE_WANDB:
    # Map run names to stages
    stage_map = {
        "gpt_125m":  "Pretrain",
        "sft_chat":  "SFT chat",
        "sft_code":  "SFT code",
        "dpo":       "DPO",
    }

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    colors = ["#378ADD", "#AFA9EC", "#7F77DD", "#D85A30"]

    for ax, (key, label), color in zip(axes, stage_map.items(), colors):
        matching = [r for r in runs if key in r.name]
        if not matching:
            ax.set_title(f"{label}\n(no run found)")
            continue
        run = matching[-1]  # latest
        history = run.history(keys=["train_loss", "val_loss"], samples=500)
        if "train_loss" in history.columns:
            ax.plot(history["_step"], history["train_loss"],
                    color=color, linewidth=1.2, label="train", alpha=0.8)
        if "val_loss" in history.columns:
            ax.plot(history["_step"], history["val_loss"],
                    color=color, linewidth=1.5, linestyle="--", label="val")
        ax.set_title(label)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.2)

    plt.suptitle("Training loss curves — all stages", fontsize=13)
    plt.tight_layout()
    plt.savefig("/results/eval/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

## Option B — Parse NeMo log files

In [ ]:
def parse_nemo_log(log_file: Path) -> dict[str, list]:
    """Extract step, train_loss, val_loss from NeMo training logs."""
    steps, train_losses, val_losses = [], [], []
    train_pat = re.compile(r"step=(\d+).*?train_loss=([\d.]+)")
    val_pat   = re.compile(r"step=(\d+).*?val_loss=([\d.]+)")

    with open(log_file) as f:
        for line in f:
            tm = train_pat.search(line)
            if tm:
                steps.append(int(tm.group(1)))
                train_losses.append(float(tm.group(2)))
            vm = val_pat.search(line)
            if vm:
                val_losses.append((int(vm.group(1)), float(vm.group(2))))

    return {"steps": steps, "train_loss": train_losses, "val_loss": val_losses}


# Find log files
log_dirs = {
    "Pretrain": RESULTS_DIR / "pretrain_logs",
    "SFT chat": RESULTS_DIR / "sft_logs",
    "SFT code": RESULTS_DIR / "sft_logs",
    "DPO":      RESULTS_DIR / "dpo_logs",
}

for label, log_dir in log_dirs.items():
    logs = sorted(log_dir.glob("*.log")) if log_dir.exists() else []
    print(f"{label:<15} {len(logs)} log file(s) in {log_dir}")

In [ ]:
# Parse and plot from log files
colors = {"Pretrain": "#378ADD", "SFT chat": "#AFA9EC", "SFT code": "#7F77DD", "DPO": "#D85A30"}
has_data = False

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (label, log_dir) in zip(axes, log_dirs.items()):
    logs = sorted(log_dir.glob("*.log")) if log_dir.exists() else []
    ax.set_title(label)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.2)

    if not logs:
        ax.text(0.5, 0.5, "No logs yet", ha="center", va="center",
                transform=ax.transAxes, color="gray")
        continue

    data = parse_nemo_log(logs[-1])
    if data["steps"]:
        ax.plot(data["steps"], data["train_loss"],
                color=colors[label], linewidth=1.2, label="train", alpha=0.8)
        has_data = True
    if data["val_loss"]:
        vsteps, vlosses = zip(*data["val_loss"])
        ax.plot(vsteps, vlosses,
                color=colors[label], linewidth=1.5, linestyle="--", label="val")
    ax.legend(fontsize=9)

plt.suptitle("Training loss curves — all stages", fontsize=13)
plt.tight_layout()

if has_data:
    plt.savefig("/results/eval/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Learning Rate Schedule

In [ ]:
# Visualize the cosine annealing LR schedule for each stage
def cosine_lr(step, max_steps, peak_lr, warmup_steps, min_lr):
    if step < warmup_steps:
        return peak_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + np.cos(np.pi * progress))

stages_lr = [
    ("Pretrain",  200000, 3e-4, 2000, 3e-5,  "#378ADD"),
    ("SFT chat",   10000, 1e-5,  100, 1e-6,  "#AFA9EC"),
    ("SFT code",   10000, 5e-6,   50, 5e-7,  "#7F77DD"),
    ("DPO",         5000, 5e-7,   50, 0.0,   "#D85A30"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for ax, (label, max_steps, peak_lr, warmup, min_lr, color) in zip(axes, stages_lr):
    steps = np.arange(max_steps)
    lrs = [cosine_lr(s, max_steps, peak_lr, warmup, min_lr) for s in steps]
    ax.plot(steps, lrs, color=color, linewidth=1.5)
    ax.set_title(label)
    ax.set_xlabel("Step")
    ax.set_ylabel("LR")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0e'))
    ax.grid(True, alpha=0.2)

plt.suptitle("Learning rate schedules (cosine annealing)", fontsize=13)
plt.tight_layout()
plt.show()